# Discover recurrent molecular events across donors

This notebook searches a rooted Gravlax collection without supplying event coordinates. Candidate discovery uses collection metadata; exact support is reduced from routed source-archive molecule records. The demo manifest supplies the donor design, cell groups, thresholds, and immutable assets. Its final figure is a deterministic SVG built from the typed result tables using only the Python standard library and the IPython display API supplied by Colab.

In [ ]:
#@title Immutable demo manifest (required)
MANIFEST_URL = "" #@param {type:"string"}
MANIFEST_SHA256 = "" #@param {type:"string"}
if not MANIFEST_URL or not MANIFEST_SHA256:
    raise RuntimeError("Demo capsule not published/configured: set the immutable manifest URL and its SHA-256. No fallback URL is used.")

In [ ]:
import hashlib, json, re, subprocess, sys, tarfile, urllib.request, zipfile
from pathlib import Path
WORK = Path('/content/gravlax-demo'); WORK.mkdir(parents=True, exist_ok=True)
HEX64 = re.compile(r'^[0-9a-f]{64}$')
def download_verified(url, sha256, destination):
    if not isinstance(url, str) or not url.startswith('https://'): raise ValueError(f'asset URL must be HTTPS, got {url!r}')
    if not isinstance(sha256, str) or not HEX64.fullmatch(sha256): raise ValueError('asset SHA-256 must be 64 lowercase hexadecimal characters')
    destination = Path(destination); temporary = destination.with_suffix(destination.suffix + '.part'); digest = hashlib.sha256()
    with urllib.request.urlopen(url) as source, temporary.open('wb') as sink:
        while block := source.read(1 << 20): digest.update(block); sink.write(block)
    if digest.hexdigest() != sha256:
        temporary.unlink(missing_ok=True); raise RuntimeError(f'SHA-256 mismatch for {url}')
    temporary.replace(destination); return destination
manifest_path = download_verified(MANIFEST_URL, MANIFEST_SHA256, WORK / 'manifest.json')
manifest = json.loads(manifest_path.read_text())
if manifest.get('schema') != 'gravlax.demo-capsule.v1': raise RuntimeError('unsupported or missing demo manifest schema')
for section in ('software', 'resources', 'stories'):
    if not isinstance(manifest.get(section), dict): raise RuntimeError(f'manifest {section} must be an object')
required_software_fields = {'version', 'aie', 'python_wheel'}
if required_software_fields.difference(manifest['software']): raise RuntimeError(f'manifest software lacks {sorted(required_software_fields.difference(manifest["software"]))}')
def fetch(spec):
    if not isinstance(spec, dict) or any(not spec.get(key) for key in ('url','sha256','filename')): raise RuntimeError(f'incomplete published asset declaration: {spec!r}')
    if Path(spec['filename']).name != spec['filename']: raise ValueError('asset filename must be a basename')
    return download_verified(spec['url'], spec['sha256'], WORK / spec['filename'])
def install_tools():
    wheel = fetch(manifest['software']['python_wheel']); subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', str(wheel)], check=True)
    spec = manifest['software']['aie']; bundle = fetch(spec); member = spec.get('member')
    if member:
        if zipfile.is_zipfile(bundle):
            with zipfile.ZipFile(bundle) as archive: payload = archive.read(member)
        else:
            with tarfile.open(bundle, 'r:*') as archive:
                item = archive.getmember(member)
                if not item.isfile(): raise RuntimeError('configured aie archive member is not a file')
                payload = archive.extractfile(item).read()
        binary = WORK / 'aie'; binary.write_bytes(payload)
    else: binary = bundle
    binary.chmod(0o755); return binary
AIE = install_tools()
from gravlax import Client, __version__ as PYTHON_VERSION
EXPECTED_VERSION = manifest['software'].get('version')
if not isinstance(EXPECTED_VERSION, str) or not EXPECTED_VERSION: raise RuntimeError('manifest software.version must be nonempty')
CLI_VERSION = subprocess.run([str(AIE), '--version'], check=True, capture_output=True, text=True).stdout.strip()
if CLI_VERSION != f'aie {EXPECTED_VERSION}': raise RuntimeError(f'CLI version mismatch: {CLI_VERSION!r} != aie {EXPECTED_VERSION}')
if PYTHON_VERSION != EXPECTED_VERSION: raise RuntimeError(f'Python version mismatch: {PYTHON_VERSION!r} != {EXPECTED_VERSION}')
client = Client(binary=AIE); print(CLI_VERSION)

In [ ]:
story = manifest['stories'].get('event_discovery'); resources = manifest['resources']
required_story_fields = {'archives', 'groups', 'design'}
if not isinstance(story, dict) or required_story_fields.difference(story): raise RuntimeError(f'event_discovery story lacks {sorted(required_story_fields.difference(story or {}))}')
if story.get('annotation') and {'assembly', 'annotation_label'}.difference(story): raise RuntimeError('event_discovery annotation requires assembly and annotation_label')
if not isinstance(story['archives'], dict) or not story['archives']: raise RuntimeError('event_discovery.archives must be a nonempty sample-to-resource object')
def story_resource(name):
    spec = resources.get(name)
    if not isinstance(name, str) or not isinstance(spec, dict): raise RuntimeError(f'story resource is not declared: {name!r}')
    return spec
archives = {sample: fetch(story_resource(resource)) for sample, resource in story['archives'].items()}
for sample, path in archives.items():
    expected = story_resource(story['archives'][sample]).get('archive_root')
    if not expected: raise RuntimeError(f'archive {sample} lacks a rooted identity in the manifest')
    identity = client.result_raw(['inspect-archive', path, '--json'])['native_identity']
    observed = f"{identity['scheme']}:{identity['blake3']}"
    if observed != expected: raise RuntimeError(f'archive root mismatch for {sample}: {observed}')
collection = WORK / 'event-discovery.aicollection'; collection.unlink(missing_ok=True)
build = ['collection', 'build']
for sample, path in sorted(archives.items()): build.append(f'--sample={sample}={path}')
if story.get('shape_routes', True): build.append('--shape-routes')
if story.get('allow_unstamped', False): build.append('--allow-unstamped')
build.extend([f'--out={collection}'])
client.run(build)
groups = fetch(story_resource(story['groups']))
design = fetch(story_resource(story['design']))
annotation = fetch(story_resource(story['annotation'])) if story.get('annotation') else None
result = client.collection_find_events(
    collection, kinds=tuple(story.get('kinds', ())), design=design, groups=groups,
    require_groups=tuple(story.get('require_groups', ())),
    min_group_umi_classes=story.get('min_group_umi_classes', 1), min_donors=story.get('min_donors', 1),
    min_samples=story.get('min_samples', 1), min_umi_classes=story.get('min_umi_classes', 1),
    min_side_umi_classes=story.get('min_side_umi_classes', 1), min_support=story.get('min_support', 2),
    terminal_cluster_bp=story.get('terminal_cluster_bp', 25),
    max_terminal_events=story.get('max_terminal_events', 10000000),
    annotation=annotation, assembly=story.get('assembly'),
    annotation_label=story.get('annotation_label'), annotation_digest=story.get('annotation_digest'),
    novel_only=story.get('novel_only', False),
    solo_strand=story.get('solo_strand', 'forward'),
    max_candidates=story.get('max_candidates', 100000),
    max_candidates_considered=story.get('max_candidates_considered', 1000000),
    max_routed_entries=story.get('max_routed_entries', 10000000),
    max_exact_match_attempts=story.get('max_exact_match_attempts', 25000000),
    max_annotation_comparisons=story.get('max_annotation_comparisons', 10000000),
)
print(json.dumps(result.summary.as_dict(), indent=2))
print('tables:', result.table_names)
required_tables = {'capabilities', 'entities', 'components', 'counts', 'terminal_anchors', 'terminal_counts'}
missing_tables = required_tables.difference(result.table_names)
if missing_tables: raise RuntimeError(f'find-events result is missing tables: {sorted(missing_tables)}')
entities = result.table('entities').records()
for entity in entities[:20]: print(entity)

In [ ]:
# Deterministic SVG generated only from the typed entities and counts tables.
from html import escape
from IPython.display import SVG, display

def require_table_columns(table, expected, label):
    missing = set(expected).difference(table.columns)
    if missing: raise RuntimeError(f'{label} table is missing columns: {sorted(missing)}')

def support_bar_svg(items, title, value_label, empty_message):
    width, label_width, plot_width, row_height = 940, 330, 500, 28
    items = items[:20]; height = 92 + row_height * max(1, len(items))
    peak = max((value for _, value in items), default=1) or 1
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" role="img" aria-label="{escape(title)}">',
             '<style>text{font-family:system-ui,sans-serif;font-size:12px}.title{font-size:16px;font-weight:600}.axis{fill:#5b6470}</style>',
             f'<text class="title" x="8" y="22">{escape(title)}</text>',
             f'<text class="axis" x="{label_width}" y="43">{escape(value_label)}</text>']
    if not items: parts.append(f'<text x="8" y="72">{escape(empty_message)}</text>')
    for index, (label, value) in enumerate(items):
        y = 58 + index * row_height; span = value * plot_width / peak
        parts.extend([f'<text x="{label_width - 8}" y="{y + 14}" text-anchor="end">{escape(str(label)[:48])}</text>',
                      f'<rect x="{label_width}" y="{y}" width="{max(span, 1)}" height="18" rx="2" fill="#3366cc"/>',
                      f'<text x="{label_width + span + 5}" y="{y + 14}">{value}</text>'])
    parts.append('</svg>'); return ''.join(parts)

entity_table, count_table = result.table('entities'), result.table('counts')
require_table_columns(entity_table, {'entity_id', 'kind', 'exact_umi_classes', 'exact_samples', 'exact_donors'}, 'entities')
require_table_columns(count_table, {'entity_id', 'donor', 'group', 'informative_umi_classes'}, 'counts')
ranked_entities = sorted(entity_table.records(), key=lambda row: (-row['exact_donors'], -row['exact_samples'], -row['exact_umi_classes'], row['entity_id']))
focus = ranked_entities[0] if ranked_entities else None
support = {}
if focus:
    for row in count_table.records():
        if row['entity_id'] == focus['entity_id']:
            label = f"{row['donor']} / {row['group']}"; support[label] = support.get(label, 0) + row['informative_umi_classes']
support_rows = sorted(support.items(), key=lambda item: (-item[1], item[0]))
title = f"Recurrent support: {focus['kind']} · {focus['entity_id']}" if focus else 'Recurrent event support'
display(SVG(data=support_bar_svg(support_rows, title, 'exact raw-UMI-value classes by donor / group', 'No event passed the manifest thresholds.')))